# Exercice 31b (cloud) : apprendre à raisonner par renforcement avec GRPO

On reprend la question de l'exercice 31a à l'envers : au lieu de **dépenser** plus de calcul à l'inférence, on **entraîne** un petit modèle (`Qwen/Qwen2.5-0.5B-Instruct`, Apache-2.0) à mieux résoudre des problèmes de GSM8K, avec une **récompense vérifiable** (1 si la réponse finale est exacte, 0 sinon) et l'algorithme **GRPO** (DeepSeekMath, 2024) de la bibliothèque TRL.

Pour chaque problème, le modèle génère un **groupe** de G = 8 réponses ; l'avantage de chaque réponse est sa récompense moins la moyenne du groupe, divisée par l'écart-type du groupe. Pas de critique (réseau de valeur) comme dans PPO : la moyenne du groupe en tient lieu.

**Perte optimisée : l'objectif GRPO** (à maximiser ; TRL minimise son opposé), avec le rapport de probabilités écrêté de PPO et une pénalité KL envers le modèle de départ :

$$\mathcal{J}(\theta) = \frac{1}{G}\sum_{i=1}^{G}\frac{1}{|o_i|}\sum_{t=1}^{|o_i|}\Big[\min\big(\rho_{i,t}\,\hat A_i,\ \mathrm{clip}(\rho_{i,t}, 1-\varepsilon, 1+\varepsilon)\,\hat A_i\big) - \beta\, \mathrm{KL}_{i,t}\Big],\qquad \hat A_i = \frac{r_i - \mathrm{moyenne}(r)}{\mathrm{écart\text{-}type}(r)}$$

Réglages : `loss_type="grpo"` (normalisation par longueur de réponse, comme dans l'article DeepSeekMath), `beta=0.04` (coefficient KL de l'article), `epsilon=0.2` (écrêtage usuel de PPO). Les **valeurs par défaut de TRL 1.15 sont autres** (voir les questions du README). Adaptateurs **LoRA** (chapitre 30) : seuls ~0,5 % des poids sont entraînés, et le modèle de référence de la pénalité KL est le même réseau avec l'adaptateur désactivé (aucune copie en mémoire).

Avant de lancer : *Exécution › Modifier le type d'exécution › GPU T4*, puis *Exécution › Tout exécuter*. Calcul en **float16** (le T4 ne fait pas de bfloat16). Seed 42. `COURT = True` (cellule 3) : 10 pas seulement, pour tester en quelques minutes.

In [ ]:
# Cellule 1 : vérifier le GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # un seul GPU visible : GRPOTrainer refuse nn.DataParallel (Kaggle « GPU T4 x2 »)
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Exécution › Modifier le type d'exécution › GPU T4, puis relancer.")
print("GPU :", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Cellule 2 : dépendances épinglées (versions vérifiées sur PyPI pour Python 3.12, 15/10/2026).
# torch n'est PAS épinglé : on garde celui de l'image Colab (le réinstaller coûterait ~1 Go).
!pip install -q trl==1.15.0 transformers==5.19.0 datasets==5.1.0 accelerate==1.15.0 peft==0.21.2

In [ ]:
# Cellule 3 : configuration
import random, re, time
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# COURT = True : 10 pas et 50 problèmes d'évaluation, pour vérifier que tout tourne (~5 min)
COURT = False
MODELE = "Qwen/Qwen2.5-0.5B-Instruct"   # Apache-2.0, non restreint (aucun token nécessaire)
config = dict(
    n_pas=10 if COURT else 150,          # pas d'optimisation GRPO
    taille_groupe=8,                     # G : réponses générées par problème
    completions_par_pas=16,              # 16 réponses par pas = 2 problèmes × 8
    tokens_max=256,                      # longueur maximale d'une réponse
    lr=2e-5,                             # LoRA : taux plus élevé qu'en ajustement complet
    beta=0.04,                           # poids de la pénalité KL (valeur de DeepSeekMath)
    epsilon=0.2,                         # écrêtage du rapport de probabilités (valeur usuelle de PPO)
    lora_r=16, lora_alpha=32,
    n_eval=50 if COURT else 200,         # problèmes du test de GSM8K pour l'évaluation
)
print(config)

In [ ]:
# Cellule 4 : données GSM8K (openai/gsm8k, licence MIT) et récompense vérifiable
from datasets import load_dataset

REVISION_GSM8K = "740312add88f781978c0658806c59bc2815b9866"
gsm8k = load_dataset("openai/gsm8k", "main", revision=REVISION_GSM8K)
CONSIGNE = ("Solve the math problem. Reason step by step, briefly, "
            "then write the final line as 'Answer: <number>'.")

def vers_exemple(ligne):
    # La solution de référence se termine par « #### 18 »
    reponse = ligne["answer"].split("####")[-1].strip().replace(",", "")
    return {"prompt": [{"role": "system", "content": CONSIGNE},
                       {"role": "user", "content": ligne["question"]}],
            "reponse": reponse}

train = gsm8k["train"].shuffle(seed=SEED).map(vers_exemple, remove_columns=["question", "answer"])
test = gsm8k["test"].shuffle(seed=SEED).select(range(config["n_eval"])).map(vers_exemple)
print(len(train), "problèmes d'entraînement ;", len(test), "d'évaluation")

NOMBRE = r"-?\d+(?:\.\d+)?"

def extraire_nombre(texte):
    # Même règle que l'exercice 31a : nombre après « Answer: », sinon dernier nombre du texte
    t = texte.replace(",", "")
    apres = re.findall(r"answer\s*:\s*[^\d\-]{0,12}(" + NOMBRE + ")", t, flags=re.IGNORECASE)
    if apres:
        return float(apres[-1])
    nombres = re.findall(NOMBRE, t)
    return float(nombres[-1]) if nombres else None

def recompense_exacte(completions, reponse, **kwargs):
    # Récompense VÉRIFIABLE : 1 si la réponse finale est exacte, 0 sinon.
    # TRL passe les colonnes du jeu de données (ici « reponse ») en arguments nommés ;
    # chaque complétion est une conversation : [{"role": "assistant", "content": …}].
    scores = []
    for c, attendu in zip(completions, reponse):
        n = extraire_nombre(c[0]["content"])
        scores.append(1.0 if n is not None and n == float(attendu) else 0.0)
    return scores

assert recompense_exacte([[{"content": "2 + 2 = 4\nAnswer: 4"}]], ["4"]) == [1.0]
assert recompense_exacte([[{"content": "Answer: 5"}]], ["4"]) == [0.0]

In [ ]:
# Cellule 5 : modèle (float16) et évaluation gloutonne avant entraînement
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELE)
tok.padding_side = "left"                 # génération par lots : remplissage à gauche
modele = AutoModelForCausalLM.from_pretrained(MODELE, dtype=torch.float16).to("cuda")

@torch.no_grad()
def evaluer(m, donnees, taille_lot=32):
    # Précision (réponse finale exacte) en décodage glouton, et longueur moyenne des réponses
    m.eval()
    justes, longueurs = 0, []
    for i in range(0, len(donnees), taille_lot):
        lot = donnees[i:i + taille_lot]
        textes = [tok.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
                  for p in lot["prompt"]]
        entree = tok(textes, return_tensors="pt", padding=True).to(m.device)
        sortie = m.generate(**entree, max_new_tokens=config["tokens_max"], do_sample=False,
                            pad_token_id=tok.pad_token_id)
        nouveaux = sortie[:, entree["input_ids"].shape[1]:]
        for ids, attendu in zip(nouveaux, lot["reponse"]):
            texte = tok.decode(ids, skip_special_tokens=True)
            n = extraire_nombre(texte)
            justes += int(n is not None and n == float(attendu))
            longueurs.append(int((ids != tok.pad_token_id).sum()))
    return justes / len(donnees), float(np.mean(longueurs))

t0 = time.perf_counter()
precision_avant, longueur_avant = evaluer(modele, test)
print(f"Avant GRPO : précision {precision_avant:.1%}, {longueur_avant:.0f} tokens par réponse "
      f"({time.perf_counter() - t0:.0f} s)")

In [ ]:
# Cellule 6 : entraînement GRPO (TRL) avec LoRA
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    output_dir="grpo-gsm8k",
    seed=SEED,
    max_steps=config["n_pas"],
    learning_rate=config["lr"],
    per_device_train_batch_size=config["completions_par_pas"],  # en complétions, pas en problèmes
    gradient_accumulation_steps=1,
    num_generations=config["taille_groupe"],
    max_completion_length=config["tokens_max"],
    temperature=1.0,                       # échantillonnage : il faut de la diversité dans le groupe
    loss_type="grpo", beta=config["beta"], epsilon=config["epsilon"],
    fp16=True,                             # jamais bf16 sur T4
    gradient_checkpointing=True,
    logging_steps=1, report_to="none", save_strategy="no",
)
lora = LoraConfig(r=config["lora_r"], lora_alpha=config["lora_alpha"], lora_dropout=0.0,
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
entraineur = GRPOTrainer(model=modele, reward_funcs=recompense_exacte, args=args,
                         train_dataset=train, processing_class=tok, peft_config=lora)
entraineur.model.print_trainable_parameters()

t0 = time.perf_counter()
entraineur.train()
duree_entrainement = time.perf_counter() - t0
print(f"Entraînement : {duree_entrainement / 60:.1f} min pour {config['n_pas']} pas")

In [ ]:
# Cellule 7 : évaluation après entraînement, même protocole qu'avant
t0 = time.perf_counter()
precision_apres, longueur_apres = evaluer(entraineur.model, test)
print(f"Avant GRPO : précision {precision_avant:.1%}, {longueur_avant:.0f} tokens par réponse")
print(f"Après GRPO : précision {precision_apres:.1%}, {longueur_apres:.0f} tokens par réponse "
      f"({time.perf_counter() - t0:.0f} s)")
print(f"Gain : {100 * (precision_apres - precision_avant):+.1f} points sur {len(test)} problèmes")

In [ ]:
# Cellule 8 : courbes d'entraînement (récompense moyenne, groupes sans signal, longueur)
import matplotlib.pyplot as plt

hist = [h for h in entraineur.state.log_history if "reward" in h]
pas = [h["step"] for h in hist]

def lisser(v, k=10):
    v = np.asarray(v, dtype=float)
    return np.convolve(v, np.ones(k) / k, mode="valid") if len(v) >= k else v

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
series = [("reward", "récompense moyenne (précision des réponses échantillonnées)"),
          ("frac_reward_zero_std", "part des groupes sans signal (écart-type nul)"),
          ("completions/mean_length", "longueur moyenne des réponses (tokens)")]
for ax, (cle, titre) in zip(axes, series):
    v = [h[cle] for h in hist]
    ax.plot(pas, v, alpha=0.3)
    ax.plot(pas[len(pas) - len(lisser(v)):], lisser(v), lw=2)
    ax.set(title=titre, xlabel="pas GRPO")
    ax.title.set_fontsize(9)
fig.tight_layout()
fig.savefig("courbes-31b.png", dpi=150)
plt.show()

In [ ]:
# Cellule 9 : une réponse du modèle entraîné, puis sauvegarde (adaptateur LoRA ≈ 9 Mo + courbes)
import json, shutil

exemple = test[0]
texte = tok.apply_chat_template(exemple["prompt"], tokenize=False, add_generation_prompt=True)
entree = tok([texte], return_tensors="pt").to("cuda")
with torch.no_grad():
    sortie = entraineur.model.generate(**entree, max_new_tokens=config["tokens_max"], do_sample=False,
                                       pad_token_id=tok.pad_token_id)
print(exemple["prompt"][1]["content"], "\n---\n",
      tok.decode(sortie[0, entree["input_ids"].shape[1]:], skip_special_tokens=True),
      "\n--- attendu :", exemple["reponse"])

entraineur.model.save_pretrained("adaptateur-grpo-31b")
bilan = dict(config=config, precision_avant=precision_avant, precision_apres=precision_apres,
             longueur_avant=longueur_avant, longueur_apres=longueur_apres,
             duree_entrainement_s=duree_entrainement, historique=hist)
with open("bilan-31b.json", "w") as f:
    json.dump(bilan, f, indent=1)
shutil.make_archive("resultats-31b", "zip", ".", "adaptateur-grpo-31b")
try:  # Colab : téléchargement direct ; Kaggle : fichiers dans /kaggle/working (onglet Output)
    from google.colab import files
    for nom in ("resultats-31b.zip", "bilan-31b.json", "courbes-31b.png"):
        files.download(nom)
except ImportError:
    print("Hors Colab : fichiers laissés dans le dossier courant.")